In [1]:
# import nessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
%matplotlib inline
df = pd.read_csv('/workspaces/customer_chrun_prediction/data/processed/clean_customer_churn_dataset.csv')
df.head(2)

,customer_id,tenure,monthly_charges,total_charges,contract,payment_method,internet_service,tech_support,online_security,support_calls,churn
0,1,52,54.20,2818.4,Month-to-month,Credit,DSL,No,Yes,1,No
1,2,15,35.28,529.2,Month-to-month,Debit,DSL,No,No,2,No


In [2]:
df.payment_method.value_counts()

payment_method
Credit    5026
Debit     5025
Cash      4995
UPI       4954
Name: count, dtype: int64

In [3]:
#----
# Feature Engineering Steps Here
#----
df['is_new_customer'] = np.where(df['tenure'] <= 6,1,0)

df['is_long_term_customer'] = np.where(df['tenure'] >= 36,1,0)

# Pringing & spend
df['monthly_charges_log'] = np.log(df['monthly_charges'] + 1)

df['high_price_flag'] = np.where(df['monthly_charges'] > 81,1,0)

df['price_to_tenure_ratio'] = df['monthly_charges'] / (df['tenure'])

# Payment Method
df['is_auto_pay']= np.where(df['payment_method'].str.contains('Credit|UPI|Debit'),1,0)

# support calls
df['high_support_calls'] = np.where(df['support_calls'] >= 3,1,0)
df['support_calls_per_month'] = df['support_calls'] / (df['tenure'])
df['recent_issue_proxxy'] = np.where((df['support_calls'] > 2) & (df['tenure'] <= 3),1,0)

# Interactions Features
df['high_price_new_customer'] = np.where((df['high_price_flag'] == 1) & (df['is_new_customer'] == 1),1,0)
df['month_to_month_high_support'] = np.where((df['contract'] == 'Month-to-month') & (df['high_support_calls'] == 1),1,0)
df['long_term_low_support'] = np.where((df['is_long_term_customer'] == 1) & (df['high_support_calls'] == 0),1,0)


df.head(20)

,customer_id,tenure,monthly_charges,total_charges,contract,payment_method,internet_service,tech_support,online_security,support_calls,...,monthly_charges_log,high_price_flag,price_to_tenure_ratio,is_auto_pay,high_support_calls,support_calls_per_month,recent_issue_proxxy,high_price_new_customer,month_to_month_high_support,long_term_low_support
0,1,52,54.20,2818.40,Month-to-month,Credit,DSL,No,Yes,1,...,4.010963,0,1.042308,1,0,0.019231,0,0,0,1
1,2,15,35.28,529.20,Month-to-month,Debit,DSL,No,No,2,...,3.591267,0,2.352000,1,0,0.133333,0,0,0,0
2,3,72,78.24,5633.28,Month-to-month,Debit,DSL,No,No,0,...,4.372481,0,1.086667,1,0,0.000000,0,0,0,1
3,4,61,80.24,4894.64,One year,Cash,Fiber,Yes,Yes,0,...,4.397408,0,1.315410,0,0,0.000000,0,0,0,1
4,5,21,39.38,826.98,Month-to-month,UPI,Fiber,No,No,4,...,3.698335,0,1.875238,1,1,0.190476,0,0,1,0
5,6,24,39.66,951.84,Two year,Cash,Fiber,No,No,1,...,3.705245,0,1.652500,0,0,0.041667,0,0,0,0
6,7,3,118.95,356.85,Month-to-month,Credit,DSL,No,No,2,...,4.787075,1,39.650000,1,0,0.666667,0,1,0,0
7,8,22,44.33,975.26,Month-to-month,Debit,Fiber,Yes,Yes,0,...,3.813969,0,2.015000,1,0,0.000000,0,0,0,0
8,9,53,103.82,5502.46,Month-to-month,UPI,Fiber,Yes,Yes,1,...,4.652245,1,1.958868,1,0,0.018868,0,0,0,1
9,10,2,67.03,134.06,Two year,Credit,DSL,No,Yes,2,...,4.219949,0,33.515000,1,0,1.000000,0,0,0,0


In [4]:
# drop unnecessary columns
df.drop(columns=['customer_id'], inplace=True)

In [5]:
# save the new dataset
df.to_csv('/workspaces/customer_chrun_prediction/data/processed/feature_engineered_customer_churn_dataset.csv', index=False)